# Phone-Level Accentedness Scoring — Colab

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/aviadarn/Accentedness-Scoring-Challenge/blob/main/notebooks/phone_accentedness_colab.ipynb)

This notebook runs the repository's promoted **E16** checkpoint, verifies its bytes before inference, scores an example through the required `score_phonemes()` API, lets you upload your own WAV recording with automatic US-English phoneme generation, and launches the Gradio demo.

> The score is a model estimate of a subjective American-English annotation target. It is not a measure of intelligence, identity, employability, nationality, fluency, or communication ability, and it must not be used for high-stakes decisions.

## Before you start

Use **Runtime → Change runtime type → T4 GPU** when available. Python 3.11 and 3.12 are supported for inference and the demo. For full scientific E18/E19 runs, select Colab runtime version **2025.07** (Python 3.11) and a GPU. Run cells in order. The production demo and inference do not need the private challenge dataset, an API key, or a Hugging Face token. Never paste secrets into this notebook.

The optional E18/E19 experiments do need the original challenge data and the train-only pseudo-speaker map, which are intentionally excluded from Git. Their cells remain off unless you explicitly enable them.

In [ ]:
from pathlib import Path

# Public checkout configuration. REPO_REF may be a branch, tag, or commit SHA.
REPO_URL = "https://github.com/aviadarn/Accentedness-Scoring-Challenge.git"
REPO_REF = "main"
CHECKOUT_DIR = Path("/content/Accentedness-Scoring-Challenge")

# Optional experiment-data overrides (for example, paths in mounted Drive).
# Leave blank to use data/dataset and data/speaker_clusters/... in the checkout.
DATA_DIR_OVERRIDE = ""
SPEAKER_MAP_OVERRIDE = ""
E16_OOF_OVERRIDE = ""  # Optional strict E18 baseline-binding artifact.
# Optional persistent archive folder after mounting Drive, for example:
# from google.colab import drive; drive.mount("/content/drive")
DRIVE_ARCHIVE_DIR = ""  # e.g. /content/drive/MyDrive/accent-score-runs

# Interactive actions. Inference runs below; the demo launches by default.
LAUNCH_DEMO = True
UPLOAD_NOW = False
RUN_E18_QUICK = False
RUN_E19_QUICK = False
RUN_E18_FULL = False
RUN_E19_FULL = False
CONFIRM_LONG_FULL_RUNS = False

EXPECTED_CHECKPOINT_SHA256 = (
    "ead3144c82ab87ad9d6406511c6348a99c944a9f8ac1097756a6a61d78e80338"
)

In [ ]:
import subprocess

def find_repository(start: Path) -> Path | None:
    start = start.resolve()
    for candidate in (start, *start.parents):
        if (candidate / "submission" / "inference.py").is_file():
            return candidate
    return None

def git_text(repository: Path, *arguments: str) -> str:
    result = subprocess.run(
        ["git", "-C", str(repository), *arguments],
        check=True, capture_output=True, text=True,
    )
    return result.stdout.strip()

def verify_existing_repository(repository: Path) -> None:
    if not (repository / ".git").exists():
        raise RuntimeError(
            f"{repository} looks like an exported source tree, not a Git checkout. "
            "Use a fresh CHECKOUT_DIR so REPO_REF can be verified."
        )
    try:
        subprocess.run(
            ["git", "-C", str(repository), "fetch", "--quiet", "--depth", "1", REPO_URL, REPO_REF],
            check=True, capture_output=True, text=True,
        )
        requested_commit = git_text(repository, "rev-parse", "FETCH_HEAD^{commit}")
        current_commit = git_text(repository, "rev-parse", "HEAD^{commit}")
        local_changes = git_text(repository, "status", "--porcelain", "--untracked-files=normal")
    except subprocess.CalledProcessError as error:
        detail = (error.stderr or error.stdout or str(error)).strip()
        raise RuntimeError(f"Could not resolve and verify REPO_REF={REPO_REF!r}: {detail}") from error
    if local_changes:
        raise RuntimeError(
            f"Existing checkout {repository} has local changes. The notebook will not silently "
            "mix them with the requested ref. Commit/stash them or use a fresh CHECKOUT_DIR."
        )
    if current_commit != requested_commit:
        raise RuntimeError(
            f"Existing checkout is stale for REPO_REF={REPO_REF!r}.\n"
            f"HEAD:      {current_commit}\nRequested: {requested_commit}\n"
            "The notebook will not change an existing checkout. Use a fresh CHECKOUT_DIR "
            "or explicitly check out the requested commit."
        )
    print(f"Verified REPO_REF={REPO_REF!r} at {current_commit}.")

existing = find_repository(Path.cwd())
if existing is not None:
    REPO_ROOT = existing.resolve()
elif (CHECKOUT_DIR / "submission" / "inference.py").is_file():
    REPO_ROOT = CHECKOUT_DIR.resolve()
else:
    if CHECKOUT_DIR.exists():
        raise RuntimeError(
            f"{CHECKOUT_DIR} exists but is not this repository. "
            "Choose a different CHECKOUT_DIR; the notebook will not overwrite it."
        )
    CHECKOUT_DIR.parent.mkdir(parents=True, exist_ok=True)
    subprocess.run(
        ["git", "clone", "--filter=blob:none", "--no-checkout", REPO_URL, str(CHECKOUT_DIR)],
        check=True,
    )
    subprocess.run(
        ["git", "-C", str(CHECKOUT_DIR), "fetch", "--depth", "1", "origin", REPO_REF],
        check=True,
    )
    subprocess.run(
        ["git", "-C", str(CHECKOUT_DIR), "checkout", "--detach", "FETCH_HEAD"],
        check=True,
    )
    REPO_ROOT = CHECKOUT_DIR.resolve()
verify_existing_repository(REPO_ROOT)
print(f"Using verified checkout: {REPO_ROOT}")

SUBMISSION_DIR = REPO_ROOT / "submission"
DATA_DIR = (
    Path(DATA_DIR_OVERRIDE).expanduser().resolve()
    if DATA_DIR_OVERRIDE
    else REPO_ROOT / "data" / "dataset"
)
SPEAKER_MAP = (
    Path(SPEAKER_MAP_OVERRIDE).expanduser().resolve()
    if SPEAKER_MAP_OVERRIDE
    else REPO_ROOT / "data" / "speaker_clusters" / "train_only_groups.json"
)
E16_OOF = Path(E16_OOF_OVERRIDE).expanduser().resolve() if E16_OOF_OVERRIDE else None

## Install the reproducible runtime

The next cell installs `ffmpeg` and `libsndfile1`, then exports exact dependency versions from `submission/uv.lock`. `--no-hashes` lets pip select the compatible wheel for the active 3.11/3.12 Colab interpreter while retaining every locked version. The cell deliberately omits the editable project wrapper (whose metadata requires Python 3.11) and adds `submission/` to `sys.path` later. It is safe to rerun. Full scientific E18/E19 runs remain restricted to Python 3.11.

In [ ]:
import importlib
import shutil
import sys

if sys.version_info[:2] not in {(3, 11), (3, 12)}:
    raise RuntimeError(
        f"Inference supports Colab Python 3.11 or 3.12, but this runtime is {sys.version.split()[0]}. "
        "Select a supported Colab runtime version."
    )

if shutil.which("apt-get"):
    missing_system = []
    for package in ("ffmpeg", "libsndfile1"):
        result = subprocess.run(
            ["dpkg-query", "-W", "-f=${Status}", package],
            capture_output=True,
            text=True,
        )
        if result.returncode != 0 or "install ok installed" not in result.stdout:
            missing_system.append(package)
    if missing_system:
        print(f"Installing system packages: {', '.join(missing_system)}")
        subprocess.run(["apt-get", "update", "-qq"], check=True)
        subprocess.run(["apt-get", "install", "-y", "-qq", *missing_system], check=True)

print("Installing the locked Python environment (the first run can take a few minutes)...")
subprocess.run(
    [sys.executable, "-m", "pip", "install", "--quiet", "uv==0.11.0"],
    check=True,
)
locked_requirements = Path("/tmp/accent-score-colab-requirements.txt")
subprocess.run(
    [
        sys.executable, "-m", "uv", "export",
        "--project", str(SUBMISSION_DIR),
        "--frozen", "--no-dev", "--no-emit-project", "--no-hashes",
        "--format", "requirements.txt",
        "--output-file", str(locked_requirements),
    ],
    check=True,
)
subprocess.run(
    [sys.executable, "-m", "pip", "install", "--quiet", "-r", str(locked_requirements)],
    check=True,
)
importlib.invalidate_caches()
print(f"Locked submission dependencies installed for Python {sys.version_info.major}.{sys.version_info.minor}.")

## Verify the production checkpoint

Inference stops before loading the model unless the deployment manifest has the expected schema and promotion status and **every** listed model-bundle file exists with its recorded SHA-256. The model weight must also match the independently fixed E16 hash. This prevents a stale branch, partial bundle, or incomplete download from silently producing different scores.

In [ ]:
import hashlib
import json
import os

model_dir = SUBMISSION_DIR / "model"
checkpoint_path = model_dir / "model.safetensors"
manifest_path = model_dir / "deployment_manifest.json"
if not manifest_path.is_file():
    raise FileNotFoundError(f"Missing deployment manifest: {manifest_path}")
deployment_manifest = json.loads(manifest_path.read_text(encoding="utf-8"))
if not isinstance(deployment_manifest, dict):
    raise RuntimeError("Deployment manifest must be a JSON object.")
if deployment_manifest.get("schema_version") != "e16-deployment-manifest-v1":
    raise RuntimeError("Unexpected deployment-manifest schema.")
if deployment_manifest.get("production_promoted") is not True or deployment_manifest.get("status") != "promoted":
    raise RuntimeError("The deployment manifest does not mark this bundle as promoted.")
deployed_files = deployment_manifest.get("deployed_checkpoint_files")
if not isinstance(deployed_files, dict) or not deployed_files:
    raise RuntimeError("Deployment manifest has no deployed checkpoint file map.")
if deployed_files.get("model.safetensors") != EXPECTED_CHECKPOINT_SHA256:
    raise RuntimeError("Deployment manifest does not bind the expected E16 model hash.")

def file_sha256(path: Path) -> str:
    digest = hashlib.sha256()
    with path.open("rb") as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()

verified_bundle = {}
for relative_name, expected_digest in sorted(deployed_files.items()):
    relative_path = Path(relative_name)
    if (
        not isinstance(relative_name, str)
        or relative_path.is_absolute()
        or len(relative_path.parts) != 1
        or relative_name in {"", ".", ".."}
        or "\\" in relative_name
    ):
        raise RuntimeError(f"Unsafe deployed file name in manifest: {relative_name!r}")
    if (
        not isinstance(expected_digest, str)
        or len(expected_digest) != 64
        or any(character not in "0123456789abcdef" for character in expected_digest)
    ):
        raise RuntimeError(f"Invalid SHA-256 in manifest for {relative_name!r}.")
    deployed_path = model_dir / relative_path
    if deployed_path.is_symlink() or not deployed_path.is_file():
        raise FileNotFoundError(f"Missing regular deployed file: {deployed_path}")
    actual_digest = file_sha256(deployed_path)
    if actual_digest != expected_digest:
        raise RuntimeError(
            f"Deployment bundle verification failed for {relative_name}.\n"
            f"Expected: {expected_digest}\nActual:   {actual_digest}"
        )
    verified_bundle[relative_name] = actual_digest
actual_sha256 = verified_bundle["model.safetensors"]
if actual_sha256 != EXPECTED_CHECKPOINT_SHA256:
    raise RuntimeError("The independently expected E16 checkpoint hash did not match.")

os.chdir(REPO_ROOT)
if str(SUBMISSION_DIR) not in sys.path:
    sys.path.insert(0, str(SUBMISSION_DIR))
os.environ["ACCENT_MODEL_DIR"] = str(SUBMISSION_DIR / "model")

import torch
runtime_device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Verified {len(verified_bundle)} promoted E16 bundle files.")
print(f"Checkpoint SHA-256: {actual_sha256}")
print(f"Inference device: {runtime_device}")

## Required API smoke test

If the local challenge audio is available, this scores `utt_2446.wav`. Otherwise the cell creates a short synthetic tone so the exact public `score_phonemes(audio_path, phonemes)` path is still exercised. A tone is **not speech**, so fallback scores confirm only that loading and inference work; they have no pronunciation meaning.

In [ ]:
import numpy as np
import soundfile as sf

from inference import score_phonemes

example_phones = ["n", "oʊ", "s", "ɝ"]
dataset_example = DATA_DIR / "audio" / "utt_2446.wav"
if dataset_example.is_file():
    example_audio = dataset_example
    example_kind = "challenge audio"
else:
    example_audio = Path("/tmp/accent_score_pipeline_smoke.wav")
    sample_rate = 16_000
    timeline = np.arange(2 * sample_rate, dtype=np.float32) / sample_rate
    synthetic = (0.08 * np.sin(2 * np.pi * 220.0 * timeline)).astype(np.float32)
    sf.write(example_audio, synthetic, sample_rate, subtype="PCM_16")
    example_kind = "synthetic pipeline-only tone"

example_scores = score_phonemes(str(example_audio), example_phones)
assert len(example_scores) == len(example_phones)
assert all(np.isfinite(score) and 0.0 <= score <= 100.0 for score in example_scores)
print(f"Scored {example_kind}: {example_audio}")
for index, (phone, score) in enumerate(zip(example_phones, example_scores, strict=True), 1):
    print(f"{index:>2}. {phone:<4} {score:6.2f}")

## Score your recording from text

The helper below converts a sentence to the challenge's phoneme vocabulary with `gruut`, validates a 0.5–30 second WAV recording, and returns one score per phone. Read the sentence exactly as written. Automatic phones are a starting point; connected speech, names, or unusual spellings may need manual review in the Gradio editor.

Set `UPLOAD_NOW = True` in the configuration cell (or immediately below) and rerun the upload cell. Upload only audio you are comfortable processing in the active Colab session.

In [ ]:
from IPython.display import Audio, Markdown, display

from accent_score.demo import inspect_audio
from accent_score.g2p import text_to_phonemes

def score_text_recording(audio_path: str | Path, text: str) -> dict:
    audio_path = Path(audio_path).expanduser().resolve()
    if not audio_path.is_file():
        raise FileNotFoundError(f"Audio file does not exist: {audio_path}")
    if audio_path.suffix.lower() != ".wav":
        raise ValueError("Upload a WAV file. The public interchange format is WAV.")
    phones = text_to_phonemes(text)
    inspection = inspect_audio(audio_path)
    scores = score_phonemes(str(audio_path), list(phones))
    if len(scores) != len(phones):
        raise RuntimeError("The scorer returned the wrong number of phone scores.")
    return {
        "audio_path": audio_path,
        "text": text,
        "duration_seconds": inspection.duration_seconds,
        "phones": phones,
        "scores": scores,
    }

def show_scoring_result(result: dict) -> None:
    mean_score = sum(result["scores"]) / len(result["scores"])
    rows = ["| Position | Phone | Score |", "|---:|:---:|---:|"]
    rows.extend(
        f"| {index} | {phone} | {score:.2f} |"
        for index, (phone, score) in enumerate(
            zip(result["phones"], result["scores"], strict=True), 1
        )
    )
    display(Markdown(
        f"**{len(result['phones'])} phones · mean {mean_score:.1f}/100 · "
        f"audio {result['duration_seconds']:.2f}s**\n\n" + "\n".join(rows)
    ))
    display(Audio(filename=str(result["audio_path"])))

def upload_and_score(text: str) -> dict:
    try:
        from google.colab import files
    except ImportError as error:
        raise RuntimeError("Interactive upload is available in Google Colab only.") from error
    uploaded = files.upload()
    if len(uploaded) != 1:
        raise ValueError(f"Upload exactly one WAV file; received {len(uploaded)}.")
    original_name, payload = next(iter(uploaded.items()))
    if len(payload) > 15 * 1024 * 1024:
        raise ValueError("The upload exceeds the 15 MB demo limit.")
    safe_name = Path(original_name).name
    if Path(safe_name).suffix.lower() != ".wav":
        raise ValueError("Upload a .wav recording.")
    upload_dir = Path("/content/accent_score_uploads")
    upload_dir.mkdir(parents=True, exist_ok=True)
    destination = upload_dir / safe_name
    destination.write_bytes(payload)
    result = score_text_recording(destination, text)
    show_scoring_result(result)
    return result

In [ ]:
SENTENCE_TO_READ = "No sir, I did not see it."

if UPLOAD_NOW:
    uploaded_result = upload_and_score(SENTENCE_TO_READ)
else:
    generated = text_to_phonemes(SENTENCE_TO_READ)
    print(f"Sentence: {SENTENCE_TO_READ}")
    print("Phones:  " + " ".join(generated))
    print("Set UPLOAD_NOW = True, then rerun this cell to choose and score a WAV.")

## Launch the Gradio demo

The app provides practice sentences, browser speech playback, microphone/upload recording, editable phones, coaching bands, and ordered per-phone scores. In Colab, Gradio creates a temporary public share URL. Anyone with that URL can reach the app while the cell is running, so do not upload sensitive audio. Stop the runtime or call `colab_demo.close()` when finished.

In [ ]:
if LAUNCH_DEMO:
    if "colab_demo" in globals():
        try:
            colab_demo.close()
        except Exception:
            pass
    from demo_app import DEMO_CSS, MAX_UPLOAD_SIZE, build_demo

    colab_demo = build_demo()
    colab_demo.launch(
        share=True,
        prevent_thread_lock=True,
        show_error=False,
        max_file_size=MAX_UPLOAD_SIZE,
        css=DEMO_CSS,
    )
else:
    print("Demo launch skipped. Set LAUNCH_DEMO = True and rerun this cell.")

## Optional training-only experiments

**E16 remains the production model.** E18 compares balanced record sampling, SpecAugment, Whisper-small, and an alignment-diagnostics ablation on leakage-safe folds. E19 evaluates nested phone-specific continuous calibration. E18/E19 are research-only: even a passing result cannot alter `submission/model/` or reopen the already-used final validation boundary.

The original data is not downloaded because its redistribution/consent status is undocumented. Mount Drive or upload it yourself, then set the override paths in the first configuration cell. Expected inputs are `train.jsonl`, its referenced `audio/` files, and `train_only_groups.json`. Full runners enforce their immutable E16 fingerprints and fail closed on different data. **Everything under Colab `/content` is ephemeral.** Mount Drive and set `DRIVE_ARCHIVE_DIR` to an existing Drive folder if you want completed experiment directories archived there automatically.

In [ ]:
def training_data_status() -> dict[str, bool]:
    return {
        "train manifest": (DATA_DIR / "train.jsonl").is_file(),
        "audio directory": (DATA_DIR / "audio").is_dir(),
        "train-only pseudo-speaker map": SPEAKER_MAP.is_file(),
    }

def require_training_data() -> None:
    status = training_data_status()
    missing = [name for name, available in status.items() if not available]
    if missing:
        raise FileNotFoundError(
            "E18/E19 data is unavailable: " + ", ".join(missing) + ".\n"
            f"DATA_DIR={DATA_DIR}\nSPEAKER_MAP={SPEAKER_MAP}\n"
            "Upload or mount the private challenge data, set DATA_DIR_OVERRIDE and "
            "SPEAKER_MAP_OVERRIDE in the configuration cell, then rerun checkout/configuration."
        )

for item, available in training_data_status().items():
    print(f"{'✓' if available else '—'} {item}")
print(f"DATA_DIR: {DATA_DIR}")
print(f"SPEAKER_MAP: {SPEAKER_MAP}")

### Bounded E18/E19 smoke runs (opt in)

Quick mode exercises the complete control flow with reduced data, epochs, and bootstrap draws. It is explicitly **not scientific evidence**. Set either `RUN_E18_QUICK` or `RUN_E19_QUICK` to `True`. The notebook chooses CUDA when available and otherwise the runner's `auto` device; it never requests MPS. Fresh Colab runtimes use `--allow-download` for the pinned Whisper revisions. A completed run is atomically ZIP-archived when `DRIVE_ARCHIVE_DIR` is configured; an existing archive is never overwritten.

In [ ]:
import shlex
import shutil
import tempfile
import zipfile

EXPERIMENT_DEVICE = "cuda" if torch.cuda.is_available() else "auto"
if EXPERIMENT_DEVICE not in {"cuda", "auto"}:
    raise RuntimeError("Colab experiments may use only CUDA or automatic device selection.")

def archive_completed_experiment(output_dir: Path) -> Path | None:
    report_path = output_dir / "report.json"
    if not report_path.is_file():
        raise FileNotFoundError(f"Cannot archive an incomplete run without report.json: {output_dir}")
    if not DRIVE_ARCHIVE_DIR:
        print("WARNING: run output is only under ephemeral /content. Set DRIVE_ARCHIVE_DIR to preserve it.")
        return None
    archive_root = Path(DRIVE_ARCHIVE_DIR).expanduser().resolve()
    if not archive_root.is_dir():
        raise FileNotFoundError(
            f"Drive archive folder does not exist: {archive_root}. Mount Drive and create it first."
        )
    resolved_output = output_dir.resolve()
    runs_root = (REPO_ROOT / "runs").resolve()
    if resolved_output == runs_root or runs_root not in resolved_output.parents:
        raise RuntimeError(f"Refusing to archive a path outside repository runs/: {resolved_output}")
    symlinks = [path for path in resolved_output.rglob("*") if path.is_symlink()]
    if symlinks:
        raise RuntimeError(f"Refusing to follow symlinks while archiving: {symlinks[0]}")
    report_digest = file_sha256(report_path)
    archive_name = f"{output_dir.parent.name}-{output_dir.name}-{report_digest[:12]}.zip"
    destination = archive_root / archive_name
    report_member = f"{resolved_output.name}/report.json"

    def verify_archive(archive_path: Path) -> None:
        with zipfile.ZipFile(archive_path, "r") as candidate_archive:
            corrupt_member = candidate_archive.testzip()
            if corrupt_member is not None:
                raise RuntimeError(f"Archive is corrupt at {corrupt_member!r}: {archive_path}")
            try:
                archived_report = candidate_archive.read(report_member)
            except KeyError as error:
                raise RuntimeError(f"Archive is missing {report_member}: {archive_path}") from error
        if hashlib.sha256(archived_report).hexdigest() != report_digest:
            raise RuntimeError(f"Archived report does not match completed run: {archive_path}")

    if destination.exists():
        if destination.is_symlink() or not destination.is_file():
            raise RuntimeError(f"Archive destination is not a regular file: {destination}")
        verify_archive(destination)
        print(f"Verified existing archive; leaving it unchanged: {destination}")
        return destination
    with tempfile.TemporaryDirectory(prefix="accent-score-archive-", dir=archive_root) as staging:
        temporary_base = Path(staging) / destination.stem
        temporary_zip = Path(shutil.make_archive(
            str(temporary_base), "zip", root_dir=resolved_output.parent, base_dir=resolved_output.name
        ))
        verify_archive(temporary_zip)
        os.replace(temporary_zip, destination)
    print(f"Archived completed run to persistent storage: {destination}")
    return destination

def run_experiment_if_enabled(label: str, enabled: bool, command: list[str], output_dir: Path) -> None:
    print(f"{label}: {shlex.join(command)}")
    if not enabled:
        print(f"{label} is disabled.")
        return
    require_training_data()
    if output_dir.exists():
        if (output_dir / "report.json").is_file():
            print(f"{label} already completed at {output_dir}; leaving it unchanged.")
            archive_completed_experiment(output_dir)
            return
        raise FileExistsError(
            f"{output_dir} already exists without a complete report. "
            "Choose a new output path; the notebook will not overwrite a partial run."
        )
    environment = os.environ.copy()
    environment["PYTHONUNBUFFERED"] = "1"
    subprocess.run(command, cwd=REPO_ROOT, env=environment, check=True)
    if not (output_dir / "report.json").is_file():
        raise RuntimeError(f"{label} exited without a complete report: {output_dir}")
    archive_completed_experiment(output_dir)

e18_quick_output = REPO_ROOT / "runs" / "E18-completion-matrix" / "quick-colab"
e19_quick_output = REPO_ROOT / "runs" / "E19-phone-calibration" / "quick-colab"
e18_quick_command = [
    sys.executable, str(REPO_ROOT / "experiments" / "E18-completion-matrix" / "run.py"),
    "--data-dir", str(DATA_DIR), "--speaker-map", str(SPEAKER_MAP),
    "--output-dir", str(e18_quick_output), "--device", EXPERIMENT_DEVICE,
    "--allow-download", "--quick",
]
e19_quick_command = [
    sys.executable, str(REPO_ROOT / "experiments" / "E19-phone-calibration" / "run.py"),
    "--data-dir", str(DATA_DIR), "--speaker-map", str(SPEAKER_MAP),
    "--output-dir", str(e19_quick_output), "--device", EXPERIMENT_DEVICE,
    "--allow-download", "--quick",
]
run_experiment_if_enabled("E18 quick smoke", RUN_E18_QUICK, e18_quick_command, e18_quick_output)
run_experiment_if_enabled("E19 quick smoke", RUN_E19_QUICK, e19_quick_command, e19_quick_output)

### Full E18/E19 protocols (strong opt in)

These runs are compute-heavy and can exceed a normal Colab session. Commands are printed but not executed unless you set the corresponding `RUN_*_FULL = True` **and** `CONFIRM_LONG_FULL_RUNS = True`. A **Python 3.11** CUDA runtime is required; select Colab runtime version **2025.07** for the reproducible environment. Use a persistent runtime/storage plan, run one experiment at a time, and set `DRIVE_ARCHIVE_DIR` because `/content` disappears when the runtime ends.

For the strongest E18 provenance check, also set `E16_OOF_OVERRIDE` to the accepted E16 `oof_predictions.npz`; when present, the runner binds the new baseline to it exactly.

In [ ]:
e18_full_output = REPO_ROOT / "runs" / "E18-completion-matrix" / "full-s314159-colab"
e19_full_output = REPO_ROOT / "runs" / "E19-phone-calibration" / "nested-s314159-seed13-colab"
e18_full_command = [
    sys.executable, str(REPO_ROOT / "experiments" / "E18-completion-matrix" / "run.py"),
    "--data-dir", str(DATA_DIR), "--speaker-map", str(SPEAKER_MAP),
    "--output-dir", str(e18_full_output), "--device", "cuda", "--allow-download",
]
if E16_OOF is not None:
    if not E16_OOF.is_file():
        raise FileNotFoundError(f"Configured E16 OOF artifact does not exist: {E16_OOF}")
    e18_full_command.extend(["--e16-oof", str(E16_OOF)])
else:
    print("E18 note: E16_OOF_OVERRIDE is blank, so optional exact score binding is omitted.")
e19_full_command = [
    sys.executable, str(REPO_ROOT / "experiments" / "E19-phone-calibration" / "run.py"),
    "--data-dir", str(DATA_DIR), "--speaker-map", str(SPEAKER_MAP),
    "--output-dir", str(e19_full_output), "--device", "cuda", "--allow-download",
]

if RUN_E18_FULL or RUN_E19_FULL:
    if not CONFIRM_LONG_FULL_RUNS:
        raise RuntimeError("Set CONFIRM_LONG_FULL_RUNS = True to authorize a full experiment.")
    if sys.version_info[:2] != (3, 11):
        raise RuntimeError(
            f"Full scientific E18/E19 requires Python 3.11; active runtime is {sys.version.split()[0]}. "
            "Select Colab runtime version 2025.07 and rerun from the top."
        )
    if not torch.cuda.is_available():
        raise RuntimeError("A CUDA Colab runtime is required for a full experiment.")

run_experiment_if_enabled("E18 full protocol", RUN_E18_FULL, e18_full_command, e18_full_output)
run_experiment_if_enabled("E19 full protocol", RUN_E19_FULL, e19_full_command, e19_full_output)

## Done

For ordinary use, the verified E16 API and Gradio demo above are the deliverables. If you ran E18/E19, verify the Drive ZIP printed by the run cell or download the complete output directory before ending the session; `/content` is ephemeral. Treat quick outputs as smoke tests and full outputs as training-only evidence pending review—not as a production checkpoint.